In [ ]:
import re

def transliterate_slovenian_to_cyrillic(text):
    # Handle digraphs first
    digraphs = {
        "lj": "ль", "Lj": "Ль", "LJ": "ЛЬ",
        "nj": "нь", "Nj": "Нь", "NJ": "НЬ"
    }
    for latin, cyril in digraphs.items():
        text = text.replace(latin, cyril)

    # Word-initial ja/je/jo/ju/ji → я/йе/йо/ю/йи
    def initial_jvowel(m):
        combo = m.group(0)
        mapping = {'ja': 'я', 'je': 'йе', 'jo': 'йо', 'ju': 'ю', 'ji': 'йи'}
        lower = combo.lower()
        result = mapping.get(lower, combo)
        return result.capitalize() if combo[0].isupper() else result

    text = re.sub(r'\b[Jj][aeioui]', initial_jvowel, text)

    # j between vowels: aju → аю, ojo → ойо, etc.
    def j_iotation(m):
        before, after = m.group(1), m.group(2)
        lower_map = {'a': 'я', 'e': 'йе', 'o': 'йо', 'u': 'ю', 'i': 'йи'}
        upper_map = {'a': 'Я', 'e': 'ЙЕ', 'o': 'Йо', 'u': 'Ю', 'i': 'ЙИ'}
        mapping = upper_map if after.isupper() else lower_map
        return before + mapping.get(after.lower(), 'й' + after)

    text = re.sub(r'([aeiouAEIOU])j([aeiouiAEIOUI])', j_iotation, text)

    # Replace standalone "j" with й (after all context-sensitive ones)
    text = re.sub(r'j', 'й', text)
    text = re.sub(r'J', 'Й', text)

    # Map other letters
    mapping = {
        'a': 'а', 'b': 'б', 'c': 'ц', 'č': 'ч', 'd': 'д', 'e': 'е', 'f': 'ф',
        'g': 'г', 'h': 'х', 'i': 'и', 'k': 'к', 'l': 'л', 'm': 'м', 'n': 'н',
        'o': 'о', 'p': 'п', 'r': 'р', 's': 'с', 'š': 'ш', 't': 'т',
        'u': 'у', 'v': 'в', 'z': 'з', 'ž': 'ж',

        'A': 'А', 'B': 'Б', 'C': 'Ц', 'Č': 'Ч', 'D': 'Д', 'E': 'Е', 'F': 'Ф',
        'G': 'Г', 'H': 'Х', 'I': 'И', 'K': 'К', 'L': 'Л', 'M': 'М', 'N': 'Н',
        'O': 'О', 'P': 'П', 'R': 'Р', 'S': 'С', 'Š': 'Ш', 'T': 'Т',
        'U': 'У', 'V': 'В', 'Z': 'З', 'Ž': 'Ж'
    }

    return ''.join(mapping.get(char, char) for char in text)

def transliterate_file(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as infile, \
         open(output_file, 'w', encoding='utf-8') as outfile:

        for line in infile:
            stripped = line.strip()

            # Transliterate value of "# text = ..."
            if stripped.startswith("# text ="):
                prefix, text = stripped.split("=", 1)
                translit_text = transliterate_slovenian_to_cyrillic(text.strip())
                outfile.write(f"{prefix.strip()} = {translit_text}\n")

            # Skip blank lines and metadata lines
            elif stripped.startswith("#") or not stripped:
                outfile.write(line)

            # Transliterate word (but not tag)
            else:
                parts = stripped.split("\t")
                if len(parts) == 2:
                    word, tag = parts
                    word_translit = transliterate_slovenian_to_cyrillic(word)
                    outfile.write(f"{word_translit}\t{tag}\n")
                else:
                    parts = stripped.split()
                    if len(parts) == 2:
                        word, tag = parts
                        word_translit = transliterate_slovenian_to_cyrillic(word)
                        outfile.write(f"{word_translit}\t{tag}\n")
                    else:
                        outfile.write(line)  # Fallback

# Run it
transliterate_file("data/test/lemmatized_sl.txt", "data/test/sl_cyrilic.txt")
transliterate_file("data/train/lemmatized_sl.txt", "data/train/sl_cyrilic.txt")
transliterate_file("data/val/lemmatized_sl.txt", "data/val/sl_cyrilic.txt")
